In [19]:
import scanpy as sc
import pandas as pd
import numpy as np

from cfgen.eval.compute_evaluation_metrics import compute_evaluation_metrics

Here we show how to compute evaluation metrics as presented in the paper for the unimodal example. For this, you need the real single-cell anndata and a generated one. 

To run the tutorial, create a folder `sample_data` with the command `mkdir sample_data` in the current notebook folder. Then download the dentategyrus sample data for the tutorial here: https://figshare.com/s/41969059573ce7b15377.

**Utils functions**

In [20]:
def add_to_dict(d, metrics):
    for metric in metrics:
        if metric not in d:
            d[metric] = [metrics[metric]]
        else:
            d[metric]+=[metrics[metric]]
    return d

Read the real data 

In [21]:
adata_real = sc.read_h5ad("./sample_data/dentategyrus_test.h5ad")

## Preprocess the real data 

In [22]:
# Bring back counts 
adata_real.X = adata_real.layers["X_counts"].copy()
# Compute HVG (don't subset)
sc.pp.highly_variable_genes(adata_real,
                            flavor="seurat_v3",
                            n_top_genes=2000,
                            layer="X_counts",
                            subset=False)
vars_rna = adata_real.var.copy()

# Pick 30 pcs
sc.pp.normalize_total(adata_real, target_sum=1e4)
sc.pp.log1p(adata_real)
sc.tl.pca(adata_real, n_comps=30)

Initialize unique cell types 

In [23]:
celltype_unique = np.unique(adata_real.obs["clusters"])  # unique cell type 
adata_real = adata_real[:, adata_real.var.highly_variable]

# Collect generated data 

RNA generated 

In [24]:
adata_generated_path = "./sample_data/generated_cells_2.h5ad"
adata_generated = sc.read_h5ad(adata_generated_path)
adata_generated.var = vars_rna
adata_generated = adata_generated[:, adata_generated.var.highly_variable]
adata_generated.obsm["X_pca"] = adata_generated.X.toarray().dot(adata_real.varm["PCs"])

/tmp/ipykernel_1190322/2535186413.py:5: ImplicitModificationWarning: Setting element `.obsm['X_pca']` of view, initializing view as actual.
  adata_generated.obsm["X_pca"] = adata_generated.X.toarray().dot(adata_real.varm["PCs"])


## Compute metrics

In [25]:
results = {}
for ct in celltype_unique:
    adata_real_ct = adata_real[adata_real.obs["clusters"]==ct]
    adata_generated_ct = adata_generated[adata_generated.obs["clusters"]==ct]

    results_rna_ct = compute_evaluation_metrics(adata_real_ct, 
                                                adata_generated_ct, 
                                                "cfgen_rna")
    

    results_rna_ct["ct"] = ct
    results_rna = add_to_dict(results, results_rna_ct)

Evaluating for cfgen_rna
Real (466, 2000)
Generated (452, 2000)
Evaluating for cfgen_rna
Real (354, 2000)
Generated (334, 2000)
Evaluating for cfgen_rna
Real (412, 2000)
Generated (398, 2000)
Evaluating for cfgen_rna
Real (48, 2000)
Generated (34, 2000)
Evaluating for cfgen_rna
Real (155, 2000)
Generated (165, 2000)
Evaluating for cfgen_rna
Real (363, 2000)
Generated (375, 2000)
Evaluating for cfgen_rna
Real (487, 2000)
Generated (486, 2000)
Evaluating for cfgen_rna
Real (414, 2000)
Generated (422, 2000)
Evaluating for cfgen_rna
Real (75, 2000)
Generated (66, 2000)
Evaluating for cfgen_rna
Real (209, 2000)
Generated (206, 2000)
Evaluating for cfgen_rna
Real (104, 2000)
Generated (105, 2000)
Evaluating for cfgen_rna
Real (223, 2000)
Generated (226, 2000)
Evaluating for cfgen_rna
Real (90, 2000)
Generated (88, 2000)
Evaluating for cfgen_rna
Real (243, 2000)
Generated (243, 2000)


## Print metrics 

In [26]:
results_df = pd.DataFrame(results)

Cell type metrics

In [27]:
results_df.mean(0)

/tmp/ipykernel_1190322/1286237380.py:1: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  results_df.mean(0)


1-Wasserstein_PCA     21.254856
2-Wasserstein_PCA     21.341503
Linear_MMD_PCA       417.186364
RBF_MMD_PCA            1.113664
dtype: float64